In [2]:
import numpy as np
import pandas as pd
import os
import sys
import time

In [3]:
import sklearn.tree
import sklearn.linear_model
import sklearn.metrics
import sklearn.ensemble

In [4]:

from pretty_print_sklearn_tree import pretty_print_sklearn_tree

In [5]:
# Plotting utils
import matplotlib
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8') # pretty matplotlib plots

import seaborn as sns
sns.set('notebook', font_scale=1.25, style='whitegrid')

# Load all data from train/valid/test

### Load training

In [6]:
x_train=pd.read_csv('data_product_reviews/x_train.csv').values
y_train=pd.read_csv('data_product_reviews/y_train.csv').values.ravel()

print("shape of x_train and y_train",x_train.shape,y_train.shape)

shape of x_train and y_train (6346, 7729) (6346,)


### Load validation set

In [7]:
x_valid=pd.read_csv('data_product_reviews/x_valid.csv').values
y_valid=pd.read_csv('data_product_reviews/y_valid.csv').values.ravel()

print("shape of x_valid and y_valid",x_valid.shape,y_valid.shape)

shape of x_valid and y_valid (792, 7729) (792,)


### Load test set

In [8]:
x_test=pd.read_csv('data_product_reviews/x_test.csv').values
y_test=pd.read_csv('data_product_reviews/y_test.csv').values.ravel()

print("shape of x_test and y_test",x_test.shape,y_test.shape)

shape of x_test and y_test (793, 7729) (793,)


### Load vocabulary as a list of strings

In [9]:
x_valid_df=pd.read_csv('data_product_reviews/x_valid.csv')
vocab_list=list(x_valid_df.columns)

print("length of vocab_list",len(vocab_list))
print("first 10 elements of vocab_list",vocab_list[:10])

length of vocab_list 7729
first 10 elements of vocab_list ['good', 'great', 'time', 'book', "don't", 'work', 'i_have', 'read', 'make', 'if_you']


### Pack training and validation sets into big arrays (so we can use sklearn's hyperparameter search tools)

In [10]:
x_train_and_valid=np.vstack([x_train,x_valid])
y_train_and_valid=np.hstack([y_train,y_valid])

print("Combined Train+Valid X shape:" , x_train_and_valid.shape)
print("Combined Train+Valid Y shape:" , y_train_and_valid.shape)


Combined Train+Valid X shape: (7138, 7729)
Combined Train+Valid Y shape: (7138,)


# Problem 1: Decision Trees

## 1A: Train a simple tree with depth 3

In [11]:
from sklearn.tree import DecisionTreeClassifier

simple_tree=DecisionTreeClassifier(
    criterion='gini',
    max_depth=3,
    min_samples_leaf=1,
    min_samples_split=2,
    random_state=101
)

### **Fit the tree**

**TODO Train on the training set** in the next coding cell

In [12]:
simple_tree.fit(x_train,y_train)

print("Training Completed")
print("Tree Depth:",simple_tree.get_depth())
print("Number of Leaves:",simple_tree.get_n_leaves())

Training Completed
Tree Depth: 3
Number of Leaves: 8


### **Figure 1: Print Tree**

Use a helper function from the starter code

In [13]:
pretty_print_sklearn_tree(simple_tree, feature_names=vocab_list)

The binary tree structure has 15 nodes.
- depth   0 has    1 nodes, of which    0 are leaves
- depth   1 has    2 nodes, of which    0 are leaves
- depth   2 has    4 nodes, of which    0 are leaves
- depth   3 has    8 nodes, of which    8 are leaves
The decision tree:  (Note: Y = 'yes' to above question; N = 'no')
Decision: X['great'] <= 0.50?
  Y Decision: X['excel'] <= 0.50?
    Y Decision: X['disappoint'] <= 0.50?
      Y Leaf: p(y=1 | this leaf) = 0.430 (1 total training examples)
      N Leaf: p(y=1 | this leaf) = 0.114 (1 total training examples)
    N Decision: X['disappoint'] <= 0.50?
      Y Leaf: p(y=1 | this leaf) = 0.903 (1 total training examples)
      N Leaf: p(y=1 | this leaf) = 0.429 (1 total training examples)
  N Decision: X['return'] <= 0.50?
    Y Decision: X['bad'] <= 0.50?
      Y Leaf: p(y=1 | this leaf) = 0.745 (1 total training examples)
      N Leaf: p(y=1 | this leaf) = 0.415 (1 total training examples)
    N Decision: X['movie'] <= 0.50?
      Y Leaf: p(y

**Is there any internal node that has two child leaf nodes corresponding to the same
sentiment class? Why would having two children predict the same class make sense?**

**Ans:**

Yes, theres is atleast one internal node whose both child leaves predict the same sentiment class

From printed tree :
```
Decision: X['great'] <= 0.50?
  Y Decision: X['excel'] <= 0.50?
    Y Decision: X['disappoint'] <= 0.50?
      Y Leaf: p(y=1 | this leaf) = 0.430 (1 total training examples)
      N Leaf: p(y=1 | this leaf) = 0.114 (1 total training examples)
```
Both Leaves here predict 0 (negative) as their p(y=1 | this leaf) is less than 0.5

Why this make sense?

- The tree objective is to reduce impurity ( Gini ) . A slpit can reduce impurity even if both resulting leaves still have the same majority class.
- such split will be useful for later purposes, like creating more subgroups (if depth allowed)
- impurity reduction and higher within leaf purity is what makes split beneficial not the change in predicted label



## 1B : Find best Decision Tree with grid search

In [29]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth':[2,8,32,128],
    'min_samples_leaf':[1,3,9],
    'random_state':[101]
}

base_tree=DecisionTreeClassifier(criterion='gini')


In [30]:
from sklearn.model_selection import PredefinedSplit

num_train=x_train.shape[0]
num_valid=x_valid.shape[0]

test_fold=np.concatenate([
    -1*np.ones(num_train,dtype=int),
    0*np.ones(num_valid,dtype=int)
])

my_splitter=PredefinedSplit(test_fold)

print("Number of Folds:",my_splitter.get_n_splits())

Number of Folds: 1


In [31]:
searcher=GridSearchCV(
    estimator=base_tree,
    param_grid=param_grid,
    scoring='balanced_accuracy',
    return_train_score=True,
    refit=False,
    cv=my_splitter,
    n_jobs=-1
)

searcher.fit(x_train_and_valid,y_train_and_valid)
print("Best Params:",searcher.best_params_)
print("Best Score:",searcher.best_score_)


Best Params: {'max_depth': 32, 'min_samples_leaf': 3, 'random_state': 101}
Best Score: 0.7323798101459631


### Build the best decision tree

**TODO Build the Best Tree on the training set** in the next coding cell



In [32]:
best_tree = base_tree.set_params(**searcher.best_params_)
best_tree.fit(x_train,y_train)

DecisionTreeClassifier(max_depth=32, min_samples_leaf=3, random_state=101)

### Interpret the best decision tree

In [33]:
pretty_print_sklearn_tree(best_tree, feature_names=vocab_list)

The binary tree structure has 817 nodes.
- depth   0 has    1 nodes, of which    0 are leaves
- depth   1 has    2 nodes, of which    0 are leaves
- depth   2 has    4 nodes, of which    0 are leaves
- depth   3 has    8 nodes, of which    0 are leaves
- depth   4 has   16 nodes, of which    6 are leaves
- depth   5 has   20 nodes, of which    5 are leaves
- depth   6 has   30 nodes, of which   14 are leaves
- depth   7 has   32 nodes, of which   18 are leaves
- depth   8 has   28 nodes, of which   12 are leaves
- depth   9 has   32 nodes, of which   11 are leaves
- depth  10 has   42 nodes, of which   21 are leaves
- depth  11 has   42 nodes, of which   23 are leaves
- depth  12 has   38 nodes, of which   17 are leaves
- depth  13 has   42 nodes, of which   21 are leaves
- depth  14 has   42 nodes, of which   25 are leaves
- depth  15 has   34 nodes, of which   18 are leaves
- depth  16 has   32 nodes, of which   14 are leaves
- depth  17 has   36 nodes, of which   23 are leaves
- dep

**Values of Max depth and Min samples leaf**

max_depth  : 32

min_samples_leaf : 3

# Problem 2: Random forest

## 2A: Train a random forest with default settings

In [19]:
simple_forest = sklearn.ensemble.RandomForestClassifier(
    n_estimators=100,
    criterion='gini',
    max_features='sqrt',
    max_depth=3,
    min_samples_leaf=1,
    min_samples_split=2,
    random_state=101
)

### Fit the forest

**TODO Train on the training set** in the next coding cell

In [20]:
simple_forest.fit(x_train,y_train)

print("Training Completed")
print("Number of Trees:",simple_forest.n_estimators)

Training Completed
Number of Trees: 100


## 2B & Table 2: Feature Importances

### Table 2
**Sample Output** (Feel free to print all words and organize them in any software)

|**Important Words**|**Unimportant Words**|
|:-:|:-:|
|I1 |  U1  |
|I2 |  U2  |
|I3 |  U3  |
|I4 |  U4  |
|I5 |  U5  |
|I6 |  U6  |
|I7 |  U7  |
|I8 |  U8  |
|I9 |  U9  |
|I0 |  U0  |

In [28]:
importances=simple_forest.feature_importances_

sorted_indices=np.argsort(importances)[::-1]

top10_indices=sorted_indices[:10]
top10_words=[(vocab_list[i],importances[i]) for i in top10_indices]

low_importance_mask=importances< 0.00001
low_indicies=np.where(low_importance_mask)[0]

rng=np.random.RandomState(101)
sample_low=rng.choice(low_indicies,size=min(10,len(low_indicies)),replace=False)
low10_words=[(vocab_list[i],importances[i]) for i in sample_low]


df=pd.DataFrame({
    "Important Words": [w for w,_ in top10_words],
    "Unimportant Words": [w for w,_ in low10_words]
})

display(df)

,Important Words,Unimportant Words
0,return,doing_the
1,excel,bought_these
2,great,mix
3,worst,not_all
4,poor,vision
5,disappoint,example_the
6,your_money,the_author's
7,i_love,abuse
8,the_best,went_on
9,don't,also_very


## 2C: Best Random Forest via grid search



This block might take 2-10 minutes.

If yours runs significantly longer, try this out on Google Colab instead.

In [41]:
from sklearn.ensemble import RandomForestClassifier

param_grid_rf={
    'max_features' : [3,10,33,100,333],
    'max_depth' : [16,32],
    'min_samples_leaf' : [1],
    'n_estimators' : [100],
    'random_state' : [101]
}


In [42]:
base_rf=RandomForestClassifier(
    criterion='gini',
    min_samples_split=2,
    bootstrap=True
)

In [43]:
rf_search=GridSearchCV(
    estimator=base_rf,
    param_grid=param_grid_rf,
    scoring='balanced_accuracy',
    return_train_score=True,
    refit=False,
    cv=my_splitter,
    n_jobs=-1
)

### Do the search!

In [44]:
rf_search.fit(x_train_and_valid,y_train_and_valid)

print("Best Params:",rf_search.best_params_)
print("Best Score:",rf_search.best_score_)

Best Params: {'max_depth': 32, 'max_features': 33, 'min_samples_leaf': 1, 'n_estimators': 100, 'random_state': 101}
Best Score: 0.8512044503419414


### Display search results

In [45]:
results=rf_search.cv_results_

for mean_score, params in zip(results['mean_test_score'],results['params']):
    print(f"{mean_score:.4f} {params}")

0.7518 {'max_depth': 16, 'max_features': 3, 'min_samples_leaf': 1, 'n_estimators': 100, 'random_state': 101}
0.8410 {'max_depth': 16, 'max_features': 10, 'min_samples_leaf': 1, 'n_estimators': 100, 'random_state': 101}
0.8424 {'max_depth': 16, 'max_features': 33, 'min_samples_leaf': 1, 'n_estimators': 100, 'random_state': 101}
0.8394 {'max_depth': 16, 'max_features': 100, 'min_samples_leaf': 1, 'n_estimators': 100, 'random_state': 101}
0.7842 {'max_depth': 16, 'max_features': 333, 'min_samples_leaf': 1, 'n_estimators': 100, 'random_state': 101}
0.7965 {'max_depth': 32, 'max_features': 3, 'min_samples_leaf': 1, 'n_estimators': 100, 'random_state': 101}
0.8436 {'max_depth': 32, 'max_features': 10, 'min_samples_leaf': 1, 'n_estimators': 100, 'random_state': 101}
0.8512 {'max_depth': 32, 'max_features': 33, 'min_samples_leaf': 1, 'n_estimators': 100, 'random_state': 101}
0.8354 {'max_depth': 32, 'max_features': 100, 'min_samples_leaf': 1, 'n_estimators': 100, 'random_state': 101}
0.8122 {'

### Build the best random forest using the best hyperparameters found in 2B

This is necessary so you have the specific best performing forest in your workspace.

Train *only* on training set (do not merge train and valid)


In [46]:
best_rf=base_rf.set_params(**rf_search.best_params_)
best_rf.fit(x_train,y_train)

print("Best RF Trained with:",best_rf.get_params())

Best RF Trained with: {'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': 32, 'max_features': 33, 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': None, 'oob_score': False, 'random_state': 101, 'verbose': 0, 'warm_start': False}


**What is the value of max features of your best forest?**

max_features = 33

**What is the maximum possible value for max features for this dataset? Why is it
beneficial to tune this hyperparameter?**

**Ans:**

Max possible value = number of total features = vocabulary size = 7729


why tune max_features:
- It controls randomness and correlation between trees
- Low max_features leads to more randomness , less overfitting to high diversity
- High max_features leads to trees being more similar , overfitting risk
Tuning will balance these things


**When fitting random forests, what is the primary tradeoff controlled by the n estimators
hyperparameter? Can you overfit by setting it to be too large? Why or why not?**

**Ans:**

n_estimator = number of trees in forest

- more trees reduce variance - more stable and accurate model
- But more trees increase computation time and memory

**Can we overfit by making n_estimators too large?**

No, Random foresrts do not overfit as you increase number of trees

beacause,

- each tree is trained on random samples
- Forest prediction is average of all trees prediction



### Table 3: Comparison of methods on the bag-of-words to sentiment classification task.

Please report **balanced accuracy** on the train, valid, and test sets, to 3 digits of precision

**Sample Output** (Feel free to print all values and organize them by hand)

|**method**|**max depth**|**num trees**|**train BAcc**|**valid BAcc**|**test BAcc**|
|:-|:-:|:-:|:-:|:-:|:-:|
|simple Tree	| 1 | 1 | 0.123	|0.456	|0.890|
|best Tree	|1 | 1 | 0.123	|0.456	|0.890|
|simple RandomForest	|1 | 1 | 0.123	|0.456	|0.890|
|best RandomForest	|1 | 1 | 0.123	|0.456	|0.890|

In [47]:
from sklearn.metrics import balanced_accuracy_score

def get_balanced_accuracy(model,name,max_depth,n_trees):
  train_bacc=balanced_accuracy_score(y_train,model.predict(x_train))
  valid_bacc=balanced_accuracy_score(y_valid,model.predict(x_valid))
  test_bacc=balanced_accuracy_score(y_test,model.predict(x_test))
  return {
      'method':name,
      'max depth':max_depth,
      'num trees':n_trees,
      'train BAcc':round(train_bacc,3),
      'valid BAcc':round(valid_bacc,3),
      'test BAcc':round(test_bacc,3)
  }

In [51]:
results=[]

results.append(get_balanced_accuracy(
    simple_tree,
    "simple Tree",max_depth=simple_tree.get_depth(),
    n_trees=1
))

results.append(get_balanced_accuracy(
    best_tree,
    "best Tree",max_depth=best_tree.get_depth(),
    n_trees=1
))

results.append(get_balanced_accuracy(
    simple_forest,
    "simple RandomForest",max_depth="3",
    n_trees=simple_forest.n_estimators
))

results.append(get_balanced_accuracy(
    best_rf,
    "best RandomForest",max_depth="32",
    n_trees=best_rf.n_estimators
))

results_df=pd.DataFrame(results)

display(results_df)

,method,max depth,num trees,train BAcc,valid BAcc,test BAcc
0,simple Tree,3,1,0.646,0.645,0.646
1,best Tree,32,1,0.877,0.732,0.749
2,simple RandomForest,3,100,0.819,0.797,0.778
3,best RandomForest,32,100,0.964,0.851,0.837


From the results above, we can clearly observe the difference in performance across four model:

- Simple Tree(max depth = 3) performs the worst among all the models with limited depth, it underfits the data and cannot generalize well
- Best Tree(max depth = 32) improves in all Balanced Accuracy metrics compared to Simple Tree with extended depth of 32. But the gap between train and valid is high (0.877 - 0.732) which indicates overfitting - the best tree is memorizing the patterns rather than generalizing them
- Simple RandomForest(max depth = 3, num trees = 100) already acheives better valid/test BAcc than any single descision tree. By averaging over multiple trees, it reduces variance and beomes more stable.
- Best RandomForest(max depth = 32, num trees = 100) performs the best and has the highest Balanced Accuracy metrics. Tuning max depth allows trees to learn more complex patterns and generalize well. while tuning num trees allows structure prevent overfitting.